# Tutorial 11 — RAG Pipeline over Scientific Literature
**Author:** Himanshu Goel | [Website](https://himanshugoel.github.io)

Retrieval-Augmented Generation (RAG) combines a vector database of documents with an LLM — the LLM answers questions grounded in retrieved context. This is transformative for scientific Q&A: instead of hallucinating, the model cites real papers.

**Pipeline overview:**
1. Fetch PubMed abstracts via the NCBI E-utilities API
2. Embed abstracts into a local ChromaDB vector store
3. At query time, retrieve the most relevant abstracts and pass them as context to an OpenAI LLM

In [ ]:
# ── Package installation ──────────────────────────────────────────────────────
# langchain          : orchestration framework (chains, prompts, runnables)
# langchain-community: third-party integrations (Chroma wrapper, HuggingFace)
# langchain-openai   : ChatOpenAI wrapper for GPT models
!pip install langchain langchain-community langchain-openai -q

# chromadb            : fast in-process vector database
# sentence-transformers: pretrained text → vector models (used for free local embeddings)
!pip install chromadb sentence-transformers -q

# requests   : HTTP calls to the NCBI PubMed E-utilities API
# python-dotenv: loads API keys from a .env file
!pip install requests python-dotenv -q

## Step 1 — Fetch PubMed abstracts

In [ ]:
import requests
import xml.etree.ElementTree as ET
from langchain.schema import Document

# ─────────────────────────────────────────────────────────────────────────────
# fetch_pubmed_abstracts
#
# Queries the NCBI PubMed E-utilities API for articles matching `query`.
# Returns a list of LangChain Document objects, each holding:
#   - page_content : article title + abstract text (fed into the LLM as context)
#   - metadata     : pmid, title, year (useful for citation / source display)
#
# NCBI allows 3 unauthenticated requests/second. For higher throughput,
# register for a free API key at https://www.ncbi.nlm.nih.gov/account/
# ─────────────────────────────────────────────────────────────────────────────
def fetch_pubmed_abstracts(query: str, max_results: int = 20) -> list[Document]:
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    # ── Step 1: Retrieve matching PubMed IDs (PMIDs) ─────────────────────────
    # esearch returns a JSON list of PMIDs for the search term
    search = requests.get(
        f"{base}esearch.fcgi",
        params={"db": "pubmed", "term": query, "retmax": max_results, "retmode": "json"}
    )
    pmids = search.json()["esearchresult"]["idlist"]
    if not pmids:
        return []   # no results for this query

    # ── Step 2: Fetch metadata + abstracts for all PMIDs in one request ───────
    # rettype=abstract + retmode=xml returns a structured PubmedArticle XML tree
    fetch = requests.get(
        f"{base}efetch.fcgi",
        params={"db": "pubmed", "id": ",".join(pmids), "rettype": "abstract", "retmode": "xml"}
    )
    root = ET.fromstring(fetch.text)

    docs = []
    for art in root.iter("PubmedArticle"):
        # itertext() flattens nested XML (e.g. italic tags inside titles)
        title    = "".join(t.itertext() for t in art.iter("ArticleTitle"))
        abstract = "".join(t.itertext() for t in art.iter("AbstractText"))
        pmid     = art.findtext(".//PMID", "")
        year     = art.findtext(".//PubDate/Year", "?")

        # Skip articles with no abstract text (e.g. editorials, corrections)
        if abstract.strip():
            docs.append(Document(
                page_content=f"Title: {title}\n\nAbstract: {abstract}",
                metadata={"pmid": pmid, "title": title, "year": year}
            ))
    return docs


# ── Run the fetch ─────────────────────────────────────────────────────────────
print("Fetching PubMed abstracts on SILCS drug discovery...")
docs = fetch_pubmed_abstracts("SILCS protein ligand binding drug discovery", max_results=15)
print(f"Fetched {len(docs)} abstracts")

# Preview first 3 results to confirm the fetch worked
for d in docs[:3]:
    print(f"  [{d.metadata['year']}] PMID {d.metadata['pmid']}: {d.metadata['title'][:70]}...")

## Step 2 — Embed and store in ChromaDB

In [ ]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Load a local embedding model ──────────────────────────────────────────────
# all-MiniLM-L6-v2 converts text → 384-dimensional dense vectors.
# It runs entirely on CPU — no API key or GPU required after the initial download.
# These vectors capture semantic meaning so that "binding affinity" and
# "binding free energy" are close in the vector space even though the words differ.
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if docs:
    # ── Build the vector store ────────────────────────────────────────────────
    # Chroma.from_documents embeds every document and inserts them in one call.
    # persist_directory saves the index to disk so we don't re-embed on re-runs.
    vectordb = Chroma.from_documents(
        docs,
        embedder,
        persist_directory="./pubmed_chroma"   # omit to use an in-memory store
    )
    print(f"Indexed {len(docs)} documents into ChromaDB")

    # ── Sanity check: semantic similarity search ──────────────────────────────
    # similarity_search embeds the query and returns the k nearest documents
    # by cosine distance — purely meaning-based, no keyword matching.
    query     = "protein-ligand binding affinity prediction"
    retrieved = vectordb.similarity_search(query, k=3)
    print(f"\nTop 3 results for: '{query}'")
    for r in retrieved:
        print(f"  [{r.metadata['year']}] {r.metadata['title'][:60]}...")
else:
    print("No docs fetched — check your internet connection")

## Step 3 — RAG Q&A chain with OpenAI

This cell wires retrieval directly to an OpenAI LLM via LangChain's `RetrievalQA` chain.

**How it works:**
1. The question is embedded and the k most similar abstracts are retrieved from ChromaDB
2. Those abstracts are injected as context into the LLM prompt
3. The LLM generates an answer grounded in the retrieved text

Set your key before running: `export OPENAI_API_KEY=sk-...`

In [ ]:
import os
from dotenv import load_dotenv

# Loads OPENAI_API_KEY from a .env file in the project root (if present).
# Falls back to the shell environment variable if no .env file is found.
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

# ── LLM setup ────────────────────────────────────────────────────────────────
# gpt-4o-mini balances cost and quality well for RAG tasks.
# temperature=0 makes responses deterministic (no randomness in token sampling).
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ── Build the RetrievalQA chain ───────────────────────────────────────────────
# chain_type="stuff" concatenates all retrieved docs into a single prompt.
# For very long retrievals use "map_reduce" or "refine" instead.
# return_source_documents=True lets us show which PMIDs grounded each answer.
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vectordb.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
)

# ── Run example scientific questions ─────────────────────────────────────────
questions = [
    "What is the SILCS method and how does it compute binding affinity?",
    "How does SILCS compare to FEP methods for protein-ligand binding?",
    "What is the application of SILCS to hERG cardiotoxicity prediction?",
]

for q in questions:
    print(f"Q: {q}")
    result = qa.invoke(q)
    # Truncate the answer for readability; remove [:300] for the full response
    print(f"A: {result['result'][:300]}...")
    # Show which PubMed papers were used to ground the answer
    print(f"Sources (PMIDs): {[d.metadata['pmid'] for d in result['source_documents']]}")
    print()

## Key takeaways
- **RAG = Retrieval + Generation**: the LLM answers using only retrieved documents, reducing hallucination
- **ChromaDB + sentence-transformers** enables free, local embedding and retrieval — no extra API calls needed
- **OpenAI GPT-4o-mini** provides fast, cost-effective generation grounded in the retrieved PubMed abstracts
- **`return_source_documents=True`** gives full provenance — you always know which papers backed each answer
- For production: add chunking for full-text articles, hybrid BM25 + semantic retrieval, and a cross-encoder re-ranker (see Tutorial 11 NICE for the full pipeline)